# Laboratorio R0 — RoadVision C5 (reconocimiento antes de descargar)

**Proyecto:** detectar incidentes viales en cámaras fijas tipo C5 (CCTV), en tiempo real, con un sistema en cascada:

1. **Detector rápido** (YOLO o similar) en cada frame.
2. **Tracker + reglas / modelo temporal** que marca posibles incidentes.
3. **Supervisor: `nvidia/LocateAnything-3B` con LoRA**, que se activa solo en los frames marcados y localiza qué pasó.

Regla de la casa: **ningún byte de datos se descarga hasta saber cuántos son, en qué formato están y cuánto cuesta traerlos.**

| Sección | Qué responde |
|---|---|
| R0.0 | ¿Dónde guardo los datos y cuánto disco tengo? |
| R0.1 | Inventario remoto (Hugging Face + Kaggle) |
| R0.2 | Extensiones, carpetas y contenido de ZIP remotos |
| R0.3 | Formato real leyendo solo 2 KB |
| R0.4 | Tiempo de descarga y disco |
| R0.5 | Muestras pequeñas de video |
| R0.6 | Resolución, FPS, duración y plan de estandarización |
| R0.7 | Frames clave para probar LocateAnything |
| R0.8 | Modelo base, GPU y presupuesto de tiempo real |
| R0.9 | Manifiesto de procedencia |
| Final | Ficha de candidatura, descartes y riesgos |

**Cómo usarlo:** corre las celdas en orden. Las celdas que descargan algo tienen una variable al inicio (por ejemplo `DESCARGAR = False`); cámbiala a `True` solo cuando quieras bajar la muestra.

**Kaggle:** en tu PC basta con correr una vez `kaggle auth login` en la terminal del venv. En **Colab** usa un token: ve a kaggle.com/settings/api → *Generate New Token*, cópialo y guárdalo en los **Secretos** de Colab (ícono de llave 🔑 en la barra izquierda) con el nombre `KAGGLE_API_TOKEN`, y activa el acceso del notebook. La celda de R0.1 lo toma de ahí.

## Instalación

Instala las dependencias en el **mismo Python del kernel**. Usa `uv` si está disponible (tu venv no trae pip) y, si no, `pip`. Después de la primera vez puedes poner `INSTALAR = False`.

In [ ]:
INSTALAR = True

import sys, shutil, subprocess

PAQUETES = ["huggingface_hub", "pandas", "requests", "opencv-python", "kaggle", "kagglehub"]

if INSTALAR:
    if shutil.which("uv"):
        cmd = ["uv", "pip", "install", "--python", sys.executable, *PAQUETES]
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q", *PAQUETES]
    resultado = subprocess.run(cmd, capture_output=True, text=True)
    print("OK" if resultado.returncode == 0 else resultado.stderr[-1500:])

OK


In [ ]:
import os
import json
import shutil
import hashlib
import zipfile
import platform
import subprocess

import requests
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)

EXT_VIDEO = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v")
EXT_IMAGEN = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

## R0.0 — Dónde guardar los datos

Cambia la carpeta con la variable de entorno `ROADVISION_DATA` **antes de abrir el notebook**, por ejemplo en PowerShell: `$env:ROADVISION_DATA = "C:\datos\roadvision"`. O escribe la ruta directo en `DIR_DATOS`.

In [ ]:
DIR_DATOS = os.path.abspath(
    os.environ.get("ROADVISION_DATA", os.path.join(os.getcwd(), "datos_roadvision"))
)
DIR_MUESTRAS = os.path.join(DIR_DATOS, "muestras")
DIR_FRAMES = os.path.join(DIR_DATOS, "frames")
os.makedirs(DIR_MUESTRAS, exist_ok=True)
os.makedirs(DIR_FRAMES, exist_ok=True)

DISCO_LIBRE_GB = round(shutil.disk_usage(DIR_DATOS).free / 1e9, 1)

print("Carpeta de datos:", DIR_DATOS)
print("Disco libre:", DISCO_LIBRE_GB, "GB")
if "onedrive" in DIR_DATOS.lower():
    print("\n⚠️  La carpeta está dentro de OneDrive: va a intentar subir cada video que descargues.\n"
          "    Usa una carpeta fuera de OneDrive, por ejemplo C:\\datos\\roadvision.")
if DISCO_LIBRE_GB < 50:
    print(f"\n⚠️  Solo quedan {DISCO_LIBRE_GB} GB libres.")

Carpeta de datos: /content/datos_roadvision
Disco libre: 75.4 GB


## R0.1 — Inventario remoto (sin descargar medios)

Funciones de inventario: Hugging Face con `HfApi`, Kaggle con el paquete `kaggle` (lista archivos y tamaños sin bajarlos; `kagglehub` no puede hacerlo).

In [ ]:
def hf_inventario(repo_id, repo_type="dataset"):
    """Inventario remoto de Hugging Face: metadatos, SIN descargar medios.
    Los tamaños desconocidos se reportan como pendientes, nunca como 0."""
    try:
        from huggingface_hub import HfApi
    except ImportError:
        return {"error": "Instala la dependencia: huggingface_hub"}
    try:
        info = HfApi().repo_info(repo_id=repo_id, repo_type=repo_type, files_metadata=True)
    except Exception as e:
        return {"error": str(e)[:400]}

    archivos = [{"archivo": s.rfilename, "bytes": getattr(s, "size", None)}
                for s in (info.siblings or [])]
    licencia = None
    tarjeta = getattr(info, "card_data", None)
    if tarjeta is not None:
        try:
            licencia = tarjeta.get("license")
        except Exception:
            pass
    return {"fuente": "huggingface", "repo": repo_id, "repo_type": repo_type,
            "revision": getattr(info, "sha", None), "licencia": licencia,
            "gated": getattr(info, "gated", None), "archivos": archivos}


def kaggle_inventario(slug):
    """Lista los archivos de un dataset de Kaggle con su tamaño, SIN descargarlos."""
    import io, contextlib
    silencio = io.StringIO()
    try:
        # silencia el texto de ayuda largo que imprime la librería si no hay sesión
        with contextlib.redirect_stdout(silencio), contextlib.redirect_stderr(silencio):
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            api.authenticate()
    except ImportError:
        return {"error": "Instala la dependencia: kaggle"}
    except BaseException as e:  # algunas versiones llaman a exit() si no hay sesión
        return {"error": "Sin sesión de Kaggle. En la terminal del venv corre: kaggle auth login "
                         f"(detalle: {str(e)[:200]})"}

    def _attr(obj, *nombres):
        for n in nombres:
            if getattr(obj, n, None) is not None:
                return getattr(obj, n)
        return None

    archivos, token = [], None
    try:
        for _ in range(500):  # tope de páginas
            try:
                res = (api.dataset_list_files(slug, page_token=token, page_size=200) if token
                       else api.dataset_list_files(slug, page_size=200))
            except TypeError:  # versiones viejas no paginan
                res = api.dataset_list_files(slug)
            for f in _attr(res, "files", "dataset_files", "datasetFiles") or []:
                b = _attr(f, "total_bytes", "totalBytes")
                archivos.append({"archivo": _attr(f, "name", "name_nullable", "ref"),
                                 "bytes": int(b) if b is not None else None})
            token = _attr(res, "next_page_token", "nextPageToken")
            if not token:
                break
    except Exception as e:
        return {"error": str(e)[:400]}

    licencia = None
    try:
        lic = _attr(api.dataset_view(slug), "license_name", "licenseName")
        licencia = str(lic) if lic else None
    except Exception:
        pass
    return {"fuente": "kaggle", "repo": slug, "repo_type": "kaggle", "revision": None,
            "licencia": licencia, "gated": None, "archivos": archivos}


def resumir_inventario(datos, prefijo=""):
    """Agrega totales. `prefijo` limita el alcance a una carpeta del repo."""
    if datos.get("error"):
        return datos
    archivos = sorted(datos["archivos"],
                      key=lambda a: a["bytes"] if a["bytes"] is not None else -1, reverse=True)
    en_alcance = [a for a in archivos if a["archivo"] and a["archivo"].startswith(prefijo)]

    def _tot(lista):
        return (sum(a["bytes"] is None for a in lista),
                sum(a["bytes"] for a in lista if a["bytes"] is not None))

    sin, conocidos = _tot(archivos)
    sin_alc, bytes_alc = _tot(en_alcance)
    return {**datos, "archivos": archivos, "prefijo": prefijo,
            "n_archivos": len(archivos), "archivos_sin_tamano": sin,
            "bytes_conocidos": conocidos, "GB_conocidos": round(conocidos / 1e9, 3),
            "n_en_alcance": len(en_alcance), "bytes_en_alcance": bytes_alc,
            "GB_en_alcance": round(bytes_alc / 1e9, 3), "en_alcance_completo": sin_alc == 0,
            "mas_grandes": archivos[:10]}

### Sesión de Kaggle

Busca el token en este orden: variable de entorno `KAGGLE_API_TOKEN`, Secretos de Colab y, si `PEDIR_TOKEN = True`, te lo pide en un campo oculto. En tu PC, si ya corriste `kaggle auth login`, no hace falta nada.

In [ ]:
PEDIR_TOKEN = True   # si no encuentra token, te lo pide (solo en Colab)

import sys

EN_COLAB = "google.colab" in sys.modules
if not os.environ.get("KAGGLE_API_TOKEN"):
    token = None
    if EN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("KAGGLE_API_TOKEN")
        except Exception:
            token = None
        if not token and PEDIR_TOKEN:
            from getpass import getpass
            token = getpass("Pega tu token de Kaggle (kaggle.com/settings/api): ").strip() or None
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token   # lo usan tanto kaggle como kagglehub

if os.environ.get("KAGGLE_API_TOKEN"):
    print("Token de Kaggle cargado.")
elif EN_COLAB:
    print("⚠️  Sin token de Kaggle: ACCIDENT no se va a poder inventariar ni descargar.")
else:
    print("Sin token en variables de entorno; se usará la sesión de `kaggle auth login` si existe.")

Token de Kaggle cargado.


Candidatos con vista tipo C5 (cámara fija). `prefijo` limita el alcance a una carpeta; vacío = todo el repo.

In [ ]:
CANDIDATOS = [
    {"nombre": "Accident-Bench (land_space)", "fuente": "huggingface",
     "repo": "Accident-Bench/Dataset", "tipo": "dataset", "prefijo": "land_space/",
     "modulo": "Incidentes (solo la carpeta de tierra)"},
    {"nombre": "VRU-Accident", "fuente": "huggingface",
     "repo": "kyh9191/VRU-Accident", "tipo": "dataset", "prefijo": "",
     "modulo": "Atropellamientos (verificar perspectiva)"},
    {"nombre": "Traffic Anomaly (NVIDIA)", "fuente": "huggingface",
     "repo": "nvidia/PhysicalAI-Traffic-Anomaly-Reasoning", "tipo": "dataset", "prefijo": "",
     "modulo": "Anomalías CCTV (solo anotaciones; videos aparte)"},
    {"nombre": "BMD-45 Bengaluru", "fuente": "huggingface",
     "repo": "iisc-aim/BMD-45", "tipo": "dataset", "prefijo": "",
     "modulo": "Detección de vehículos en CCTV fijo (imágenes)"},
    {"nombre": "ACCIDENT (CVPR 2026)", "fuente": "kaggle",
     "repo": "picekl/accident", "tipo": "kaggle", "prefijo": "",
     "modulo": "Incidentes CCTV fijo, reales + sintéticos"},
    {"nombre": "LocateAnything-3B", "fuente": "huggingface",
     "repo": "nvidia/LocateAnything-3B", "tipo": "model", "prefijo": "",
     "modulo": "Modelo base del LoRA (NO dataset)"},
]

detalles = {}
filas = []
for c in CANDIDATOS:
    crudo = (kaggle_inventario(c["repo"]) if c["fuente"] == "kaggle"
             else hf_inventario(c["repo"], repo_type=c["tipo"]))
    d = resumir_inventario(crudo, c["prefijo"])
    d["nombre"] = c["nombre"]
    d["es_dataset"] = c["tipo"] != "model"
    d.setdefault("fuente", c["fuente"])
    detalles[c["repo"]] = d

    if d.get("error"):
        estado = "ERROR: " + d["error"][:110]
    elif d["archivos_sin_tamano"]:
        estado = "Tamaño incompleto"
    else:
        estado = "Metadatos OK"
    filas.append({
        "Nombre": c["nombre"], "Fuente": c["fuente"], "Repositorio": c["repo"],
        "Módulo": c["modulo"], "Revisión": (d.get("revision") or "")[:10],
        "Licencia": d.get("licencia"), "Gated": d.get("gated"),
        "Archivos": d.get("n_archivos"), "GB_total": d.get("GB_conocidos"),
        "Prefijo": c["prefijo"] or "(todo)", "Archivos_en_alcance": d.get("n_en_alcance"),
        "GB_en_alcance": d.get("GB_en_alcance"), "Sin_tamaño": d.get("archivos_sin_tamano"),
        "Estado": estado,
    })

tabla_inventario = pd.DataFrame(filas)
display(tabla_inventario)

,Nombre,Fuente,Repositorio,Módulo,Revisión,Licencia,Gated,Archivos,GB_total,Prefijo,Archivos_en_alcance,GB_en_alcance,Sin_tamaño,Estado
0,Accident-Bench (land_space),huggingface,Accident-Bench/Dataset,Incidentes (solo la carpeta de tierra),b27126e81c,None,False,2325,42.643,land_space/,1810,11.435,0,Metadatos OK
1,VRU-Accident,huggingface,kyh9191/VRU-Accident,Atropellamientos (verificar perspectiva),6e81d9788b,apache-2.0,False,19,3.064,(todo),19,3.064,0,Metadatos OK
2,Traffic Anomaly (NVIDIA),huggingface,nvidia/PhysicalAI-Traffic-Anomaly-Reasoning,Anomalías CCTV (solo anotaciones; videos aparte),555e2ae85b,cc-by-4.0,False,21,0.059,(todo),21,0.059,0,Metadatos OK
3,BMD-45 Bengaluru,huggingface,iisc-aim/BMD-45,Detección de vehículos en CCTV fijo (imágenes),3e3f6a0b48,cc-by-4.0,False,45996,153.219,(todo),45996,153.219,0,Metadatos OK
4,ACCIDENT (CVPR 2026),kaggle,picekl/accident,"Incidentes CCTV fijo, reales + sintéticos",,None,None,6452,20.159,(todo),6452,20.159,0,Metadatos OK
5,LocateAnything-3B,huggingface,nvidia/LocateAnything-3B,Modelo base del LoRA (NO dataset),c32291ca5e,other,False,47,7.796,(todo),47,7.796,0,Metadatos OK


In [ ]:
# Los 10 archivos más grandes de cada repo. Esto NO descarga ninguno.
filas = []
for repo, d in detalles.items():
    for a in d.get("mas_grandes", []):
        filas.append({"Repositorio": repo, "Archivo": a["archivo"],
                      "GB": round(a["bytes"] / 1e9, 3) if a["bytes"] is not None else None})
display(pd.DataFrame(filas))

,Repositorio,Archivo,GB
0,Accident-Bench/Dataset,water_space/long/videos/water_space_long_25.mp4,1.075
1,Accident-Bench/Dataset,water_space/long/videos/water_space_long_21.mp4,1.075
2,Accident-Bench/Dataset,water_space/long/videos/water_space_long_18.mp4,1.074
3,Accident-Bench/Dataset,water_space/long/videos/water_space_long_2.mp4,0.885
4,Accident-Bench/Dataset,water_space/long/videos/water_space_long_23.mp4,0.865
5,Accident-Bench/Dataset,water_space/long/videos/water_space_long_3.mp4,0.859
6,Accident-Bench/Dataset,water_space/long/videos/water_space_long_8.mp4,0.719
7,Accident-Bench/Dataset,water_space/long/videos/water_space_long_5.mp4,0.670
8,Accident-Bench/Dataset,water_space/long/videos/water_space_long_20.mp4,0.631
9,Accident-Bench/Dataset,water_space/long/videos/water_space_long_13.mp4,0.580


**Cómo leer R0.1:**

- `GB_en_alcance` es lo que de verdad vamos a usar. En Accident-Bench solo cuenta `land_space/`; `air_space/` y `water_space/` quedan fuera.
- **Traffic Anomaly (NVIDIA)** solo trae anotaciones (~100 MB). Los videos (~150 GB, mayormente UCF-Crime) se bajan aparte con los scripts del repo.
- **BMD-45** son **imágenes**, no video: sirve para entrenar y evaluar el detector (paso 1), no para el LoRA temporal.
- **LocateAnything-3B** es el modelo: sus GB son pesos, no datos.
- Si Kaggle marca "Sin sesión", corre `kaggle auth login`. Si un repo de Hugging Face es *gated*, acepta los términos en la web y corre `huggingface-cli login`.

## R0.2 — Estructura: extensiones, carpetas y ZIP remotos

In [ ]:
filas = []
for repo, d in detalles.items():
    if d.get("error") or not d["es_dataset"]:
        continue
    conteo = {}
    for a in d["archivos"]:
        if not (a["archivo"] or "").startswith(d["prefijo"]):
            continue
        ext = os.path.splitext(a["archivo"])[1].lower() or "(sin extensión)"
        cn = conteo.setdefault(ext, {"archivos": 0, "bytes": 0, "sin": 0})
        cn["archivos"] += 1
        if a["bytes"] is None:
            cn["sin"] += 1
        else:
            cn["bytes"] += a["bytes"]
    for ext, cn in conteo.items():
        filas.append({"Repositorio": repo, "Extensión": ext, "Archivos": cn["archivos"],
                      "GB": round(cn["bytes"] / 1e9, 3), "Sin_tamaño": cn["sin"]})

tabla_extensiones = pd.DataFrame(filas)
if not tabla_extensiones.empty:
    tabla_extensiones = tabla_extensiones.sort_values(["Repositorio", "GB"], ascending=[True, False])
display(tabla_extensiones)

,Repositorio,Extensión,Archivos,GB,Sin_tamaño
0,Accident-Bench/Dataset,.mp4,1737,11.426,0
1,Accident-Bench/Dataset,.json,54,0.009,0
2,Accident-Bench/Dataset,.gz,1,0.000,0
3,Accident-Bench/Dataset,.md,18,0.000,0
17,iisc-aim/BMD-45,.png,45990,153.134,0
15,iisc-aim/BMD-45,.json,2,0.063,0
16,iisc-aim/BMD-45,.jsonl,2,0.022,0
18,iisc-aim/BMD-45,.md,1,0.000,0
19,iisc-aim/BMD-45,(sin extensión),1,0.000,0
4,kyh9191/VRU-Accident,.zip,1,3.060,0


### ¿En qué carpetas están los videos e imágenes sueltos?

Cambia `NIVEL_CARPETA` para ver más o menos profundidad.

In [ ]:
NIVEL_CARPETA = 2

filas = []
for repo, d in detalles.items():
    if d.get("error") or not d["es_dataset"]:
        continue
    grupos = {}
    for a in d["archivos"]:
        nombre = (a["archivo"] or "").lower()
        if nombre.endswith(EXT_VIDEO):
            tipo = "video"
        elif nombre.endswith(EXT_IMAGEN):
            tipo = "imagen"
        else:
            continue
        carpeta = "/".join(a["archivo"].split("/")[:-1][:NIVEL_CARPETA]) or "(raíz)"
        g = grupos.setdefault((carpeta, tipo), {"n": 0, "bytes": 0, "ejemplo": a["archivo"]})
        g["n"] += 1
        g["bytes"] += a["bytes"] or 0
    for (carpeta, tipo), g in grupos.items():
        filas.append({"Repositorio": repo, "Carpeta": carpeta, "Tipo": tipo, "Archivos": g["n"],
                      "GB": round(g["bytes"] / 1e9, 3), "Ejemplo": g["ejemplo"]})

tabla_carpetas = pd.DataFrame(filas)
if not tabla_carpetas.empty:
    tabla_carpetas = tabla_carpetas.sort_values(["Repositorio", "Archivos"], ascending=[True, False])
display(tabla_carpetas)

,Repositorio,Carpeta,Tipo,Archivos,GB,Ejemplo
9,Accident-Bench/Dataset,land_space/short,video,1500,0.861,land_space/short/videos/000377.mp4
3,Accident-Bench/Dataset,land_space/medium,video,158,6.684,land_space/medium/videos/qmsQ-obL1Z4.mp4
5,Accident-Bench/Dataset,air_space/long,video,97,1.937,air_space/long/videos/air_space_long_4.mp4
4,Accident-Bench/Dataset,water_space/medium,video,81,5.727,water_space/medium/videos/water_space_medium_c_45.avi
2,Accident-Bench/Dataset,land_space/long,video,79,3.881,land_space/long/videos/qmsQ-obL1Z4_multi.mp4
6,Accident-Bench/Dataset,air_space/medium,video,56,0.497,air_space/medium/videos/air_space_medium_16.mp4
7,Accident-Bench/Dataset,air_space/short,video,53,0.258,air_space/short/videos/air_space_short_5.mp4
1,Accident-Bench/Dataset,water_space/short,video,52,8.974,water_space/short/video_2/water_space_short_18.mp4
0,Accident-Bench/Dataset,water_space/long,video,26,13.804,water_space/long/videos/water_space_long_25.mp4
8,Accident-Bench/Dataset,figures,imagen,7,0.007,figures/land-space-examples.png


### ¿Qué hay dentro de los ZIP, sin bajarlos?

Un ZIP guarda su **índice al final del archivo**. Con acceso aleatorio (`HfFileSystem` pide solo los rangos de bytes necesarios) podemos leer ese índice y listar los videos de adentro bajando apenas unos MB. Es la misma idea de *layout + índice + lecturas por rango* que hace posibles Parquet y los Cloud-Optimized GeoTIFF.

In [ ]:
def _ruta_hf(repo, archivo, revision=None):
    return f"datasets/{repo}@{revision}/{archivo}" if revision else f"datasets/{repo}/{archivo}"


def listar_zip_hf(repo, archivo, revision=None):
    """Lista los miembros de un ZIP remoto de Hugging Face leyendo solo su índice."""
    from huggingface_hub import HfFileSystem
    fs = HfFileSystem()
    with fs.open(_ruta_hf(repo, archivo, revision), "rb", block_size=1024 * 1024) as fh:
        with zipfile.ZipFile(fh) as z:
            return [{"miembro": i.filename, "bytes": i.file_size, "comprimido": i.compress_size}
                    for i in z.infolist() if not i.is_dir()]


def extraer_de_zip_hf(repo, archivo, miembros, destino, revision=None):
    """Extrae SOLO los miembros indicados de un ZIP remoto."""
    from huggingface_hub import HfFileSystem
    fs = HfFileSystem()
    rutas = []
    with fs.open(_ruta_hf(repo, archivo, revision), "rb", block_size=1024 * 1024) as fh:
        with zipfile.ZipFile(fh) as z:
            for m in miembros:
                rutas.append(z.extract(m, path=destino))
    return rutas


contenido_zips = {}
filas = []
for repo, d in detalles.items():
    if d.get("error") or not d["es_dataset"] or d["fuente"] != "huggingface":
        continue
    zips = [a["archivo"] for a in d["archivos"]
            if a["archivo"].lower().endswith(".zip") and a["archivo"].startswith(d["prefijo"])][:5]
    for z in zips:
        try:
            miembros = listar_zip_hf(repo, z, d["revision"])
            contenido_zips[(repo, z)] = miembros
            videos = [m for m in miembros if m["miembro"].lower().endswith(EXT_VIDEO)]
            exts = pd.Series([os.path.splitext(m["miembro"])[1].lower() or "(sin)" for m in miembros]
                             ).value_counts().head(5).to_dict()
            filas.append({"Repositorio": repo, "ZIP": z, "Miembros": len(miembros),
                          "Videos": len(videos),
                          "GB_descomprimido": round(sum(m["bytes"] for m in miembros) / 1e9, 3),
                          "Extensiones": exts,
                          "Ejemplo": (videos or miembros or [{"miembro": None}])[0]["miembro"]})
        except Exception as e:
            filas.append({"Repositorio": repo, "ZIP": z, "Ejemplo": "ERROR: " + str(e)[:150]})

display(pd.DataFrame(filas))

,Repositorio,ZIP,Miembros,Videos,GB_descomprimido,Extensiones,Ejemplo
0,kyh9191/VRU-Accident,VRU_videos.zip,1000,1000,3.202,{'.mp4': 1000},VRU_videos/CAP_DATA/VRU_1.mp4


## R0.3 — Formato real leyendo solo 2 KB

La extensión puede mentir; los primeros bytes (*magic bytes*) no.

In [ ]:
FIRMAS = [
    (b"PAR1", "Parquet (columnar)"),
    (b"ARROW1", "Arrow IPC / Feather v2"),
    (b"\x93NUMPY", "NumPy .npy"),
    (b"\x89HDF\r\n\x1a\n", "HDF5"),
    (b"PK\x03\x04", "ZIP (o basado en ZIP)"),
    (b"\x1f\x8b", "gzip (.gz, .tar.gz)"),
    (b"\x28\xb5\x2f\xfd", "Zstandard (.zst)"),
    (b"BZh", "bzip2"),
    (b"\xfd7zXZ\x00", "xz"),
    (b"7z\xbc\xaf\x27\x1c", "7-Zip"),
    (b"\x1a\x45\xdf\xa3", "Matroska (.mkv) / WebM"),
    (b"\x89PNG\r\n\x1a\n", "PNG"),
    (b"\xff\xd8\xff", "JPEG"),
]


def _es_texto(b):
    try:
        b.decode("utf-8")
        return True
    except UnicodeDecodeError:
        return False


def identificar_formato(cabecera: bytes) -> str:
    """Identifica un formato a partir de sus primeros bytes."""
    if len(cabecera) >= 12 and cabecera[4:8] == b"ftyp":
        marca = cabecera[8:12].decode("latin-1").strip()
        return "QuickTime (.mov)" if marca == "qt" else f"MP4 / ISO-BMFF (marca '{marca}')"
    if cabecera.startswith(b"RIFF") and len(cabecera) >= 12:
        sub = cabecera[8:12]
        return {b"AVI ": "AVI", b"WEBP": "WebP"}.get(sub, f"RIFF ({sub!r})")
    if len(cabecera) >= 262 and cabecera[257:262] == b"ustar":
        return "TAR (sin comprimir; típico de WebDataset)"
    for firma, nombre in FIRMAS:
        if cabecera.startswith(firma):
            if firma == b"\x1a\x45\xdf\xa3" and b"webm" in cabecera[:64]:
                return "WebM"
            return nombre
    if _es_texto(cabecera[:512]):
        lineas = [l for l in cabecera.decode("utf-8", errors="replace").splitlines() if l.strip()]
        primera = lineas[0] if lineas else ""
        if primera.startswith("{") and len(lineas) > 1 and lineas[1].startswith("{"):
            return "JSON Lines (un objeto por línea)"
        if primera.startswith(("{", "[")):
            return "JSON"
        if "\t" in primera:
            return f"TSV ({primera.count(chr(9)) + 1} columnas en la 1a línea)"
        if "," in primera:
            return f"CSV ({primera.count(',') + 1} columnas en la 1a línea)"
        return "Texto plano (formato no reconocido)"
    return "Binario no identificado"


def sniff_hf(repo_id, archivo, repo_type="dataset", n_bytes=2048, timeout=30):
    """Descarga SOLO los primeros n_bytes de un archivo de Hugging Face."""
    from huggingface_hub import hf_hub_url
    from huggingface_hub.utils import build_hf_headers
    url = hf_hub_url(repo_id, archivo, repo_type=repo_type)
    cabeceras = build_hf_headers()
    cabeceras["Range"] = f"bytes=0-{n_bytes - 1}"
    with requests.get(url, headers=cabeceras, stream=True, timeout=timeout, allow_redirects=True) as r:
        r.raise_for_status()
        cabecera = r.raw.read(n_bytes, decode_content=False)
        rango_ok = r.status_code == 206
    return {"repo": repo_id, "archivo": archivo, "rango_respetado": rango_ok,
            "bytes_leidos": len(cabecera), "formato_detectado": identificar_formato(cabecera),
            "primeros_bytes_hex": cabecera[:16].hex(" ")}


# Sondea los 3 archivos más grandes en alcance de cada dataset de Hugging Face.
filas = []
for repo, d in detalles.items():
    if d.get("error") or not d["es_dataset"] or d["fuente"] != "huggingface":
        continue
    for a in [x for x in d["archivos"] if x["archivo"].startswith(d["prefijo"])][:3]:
        try:
            filas.append(sniff_hf(repo, a["archivo"]))
        except Exception as e:
            filas.append({"repo": repo, "archivo": a["archivo"],
                          "formato_detectado": "ERROR: " + str(e)[:150]})
display(pd.DataFrame(filas))

,repo,archivo,rango_respetado,bytes_leidos,formato_detectado,primeros_bytes_hex
0,Accident-Bench/Dataset,land_space/long/videos/qmsQ-obL1Z4_multi.mp4,True,2048,MP4 / ISO-BMFF (marca 'mp42'),00 00 00 18 66 74 79 70 6d 70 34 32 00 00 00 00
1,Accident-Bench/Dataset,land_space/medium/videos/qmsQ-obL1Z4.mp4,True,2048,MP4 / ISO-BMFF (marca 'mp42'),00 00 00 18 66 74 79 70 6d 70 34 32 00 00 00 00
2,Accident-Bench/Dataset,land_space/medium/videos/vnuwuu_9kgM.mp4,True,2048,MP4 / ISO-BMFF (marca 'mp42'),00 00 00 18 66 74 79 70 6d 70 34 32 00 00 00 00
3,kyh9191/VRU-Accident,VRU_videos.zip,True,2048,ZIP (o basado en ZIP),50 4b 03 04 14 00 00 00 00 00 14 90 da 5a 00 00
4,kyh9191/VRU-Accident,MANUAL_DATA_VQA_annotation.json,True,2048,JSON,7b 0d 0a 20 20 22 2e 2f 56 52 55 5f 76 69 64 65
5,kyh9191/VRU-Accident,CAP_DATA_VQA_annotation.json,True,2048,JSON,7b 0d 0a 20 20 22 2e 2f 56 52 55 5f 76 69 64 65
6,nvidia/PhysicalAI-Traffic-Anomaly-Reasoning,train/causal_linkage.json,True,2048,JSON,7b 0a 20 20 22 66 6f 72 6d 61 74 22 3a 20 22 74
7,nvidia/PhysicalAI-Traffic-Anomaly-Reasoning,train/bcq_openended.json,True,2048,JSON,7b 0a 20 20 22 66 6f 72 6d 61 74 22 3a 20 22 74
8,nvidia/PhysicalAI-Traffic-Anomaly-Reasoning,train/bcq.json,True,2048,JSON,7b 0a 20 20 22 66 6f 72 6d 61 74 22 3a 20 22 74
9,iisc-aim/BMD-45,BMD-45-Train/_annotations.coco.json,True,2048,JSON,7b 22 69 6e 66 6f 22 3a 20 7b 22 64 65 73 63 72


## R0.4 — Tiempo de descarga y disco

Hugging Face y Kaggle no cobran la descarga: el costo de egreso es **0**, pero se **declara**. Lo que sí cuesta es tiempo y disco.

In [ ]:
def plan_de_traslado(bytes_totales, anchos_banda_mbps=(50, 100, 500)):
    """Traduce un tamaño a horas de descarga para varios anchos de banda."""
    fila = {"GB": round(bytes_totales / 1e9, 2), "costo_egreso_MXN": 0.0}
    for mbps in anchos_banda_mbps:
        fila[f"horas_{mbps}Mbps"] = round((bytes_totales * 8) / (mbps * 1e6) / 3600, 2)
    return fila


filas, total = [], 0
for repo, d in detalles.items():
    if d.get("error") or not d["es_dataset"]:
        continue
    fila = {"Repositorio": repo, "Alcance": d["prefijo"] or "(todo)"}
    fila.update(plan_de_traslado(d["bytes_en_alcance"]))
    fila["tamaño_completo"] = d["en_alcance_completo"]
    filas.append(fila)
    total += d["bytes_en_alcance"]

ref = {"Repositorio": "TOTAL en alcance", "Alcance": "", "tamaño_completo": None}
ref.update(plan_de_traslado(total))
filas.append(ref)
display(pd.DataFrame(filas))

gb_total = round(total / 1e9, 1)
cabe = "✅ cabe" if gb_total < DISCO_LIBRE_GB * 0.8 else "❌ no cabe con margen"
print(f"Todo lo que está en alcance suma {gb_total} GB contra {DISCO_LIBRE_GB} GB libres: {cabe}.")
print("(Sin contar los ~150 GB de videos fuente de Traffic Anomaly, que no están en su repo.)")

,Repositorio,Alcance,GB,costo_egreso_MXN,horas_50Mbps,horas_100Mbps,horas_500Mbps,tamaño_completo
0,Accident-Bench/Dataset,land_space/,11.44,0.0,0.51,0.25,0.05,True
1,kyh9191/VRU-Accident,(todo),3.06,0.0,0.14,0.07,0.01,True
2,nvidia/PhysicalAI-Traffic-Anomaly-Reasoning,(todo),0.06,0.0,0.00,0.00,0.00,True
3,iisc-aim/BMD-45,(todo),153.22,0.0,6.81,3.40,0.68,True
4,picekl/accident,(todo),20.16,0.0,0.90,0.45,0.09,True
5,TOTAL en alcance,,187.94,0.0,8.35,4.18,0.84,None


Todo lo que está en alcance suma 187.9 GB contra 75.4 GB libres: ❌ no cabe con margen.
(Sin contar los ~150 GB de videos fuente de Traffic Anomaly, que no están en su repo.)


## R0.5 — Muestras pequeñas de video

Ahora sí bajamos bytes, **pocos y a propósito**, espaciados a lo largo de cada fuente para no quedarnos solo con los primeros archivos. Hay tres caminos según cómo esté guardado el dataset (ver R0.2). Nada se descarga hasta que pongas `DESCARGAR = True` en la celda correspondiente.

In [ ]:
# Filtros comunes para las tres fuentes
# Rutas que contengan esto se ignoran. Usa nombres de carpeta completos:
# "space" a secas también borraría land_space/.
EXCLUIR = ["air_space/", "water_space/", "dashcam"]
MAX_MB_POR_VIDEO = 100


def muestra_espaciada(candidatos, n):
    """Toma n elementos repartidos uniformemente a lo largo de la lista."""
    if n <= 0 or not candidatos:
        return []
    paso = max(1, len(candidatos) // n)
    return candidatos[::paso][:n]


def pasa_filtro(ruta, excluir=EXCLUIR):
    r = (ruta or "").lower()
    return not any(t in r for t in excluir)


limite_bytes = MAX_MB_POR_VIDEO * 1e6

repos_sueltos = [r for r, d in detalles.items()
                 if not d.get("error") and d["es_dataset"] and d["fuente"] == "huggingface"
                 and any(a["archivo"].lower().endswith(EXT_VIDEO) and a["archivo"].startswith(d["prefijo"])
                         for a in d["archivos"])]
zips_con_video = [(r, z) for (r, z), m in contenido_zips.items()
                  if any(x["miembro"].lower().endswith(EXT_VIDEO) for x in m)]
repos_kaggle = [r for r, d in detalles.items()
                if not d.get("error") and d["fuente"] == "kaggle"
                and any((a["archivo"] or "").lower().endswith(EXT_VIDEO) for a in d["archivos"])]

print("Hugging Face con videos sueltos:", repos_sueltos)
print("ZIP con videos:", zips_con_video)
print("Kaggle con videos:", repos_kaggle)

Hugging Face con videos sueltos: ['Accident-Bench/Dataset']
ZIP con videos: [('kyh9191/VRU-Accident', 'VRU_videos.zip')]
Kaggle con videos: ['picekl/accident']


### A) Hugging Face, videos sueltos

In [ ]:
DESCARGAR = True
REPO = repos_sueltos[0] if repos_sueltos else None   # o escribe el repo, p. ej. "Accident-Bench/Dataset"
N_VIDEOS = 10

if DESCARGAR and REPO:
    from huggingface_hub import hf_hub_download
    d = detalles[REPO]
    cand = sorted((a for a in d["archivos"]
                   if a["archivo"].lower().endswith(EXT_VIDEO) and a["archivo"].startswith(d["prefijo"])
                   and a["bytes"] is not None and a["bytes"] <= limite_bytes and pasa_filtro(a["archivo"])),
                  key=lambda a: a["archivo"])
    destino = os.path.join(DIR_MUESTRAS, REPO.replace("/", "__"))
    bajados = [hf_hub_download(repo_id=REPO, filename=a["archivo"], repo_type="dataset",
                               revision=d["revision"], local_dir=destino)
               for a in muestra_espaciada(cand, N_VIDEOS)]
    print(f"{len(bajados)} videos descargados de {len(cand)} candidatos en {destino}")
else:
    print("Sin descargar. Pon DESCARGAR = True para bajar la muestra.")

land_space/long/videos/-GpvLzopst8.mp4: reconstructing file:   0%|          |  0.00B / 3.83MB            

land_space/long/videos/-GpvLzopst8.mp4: downloading bytes:           |  0.00B            

land_space/medium/videos/bXaAw1730zE.mp4: reconstructing file:   0%|          |  0.00B / 63.9MB            

land_space/medium/videos/bXaAw1730zE.mp4: downloading bytes:           |  0.00B            

land_space/short/videos/000140.mp4: reconstructing file:   0%|          |  0.00B /  520kB            

land_space/short/videos/000140.mp4: downloading bytes:           |  0.00B            

... (11 outputs left out)

land_space/short/videos/001154.mp4: downloading bytes:           |  0.00B            

land_space/short/videos/001323.mp4: reconstructing file:   0%|          |  0.00B /  360kB            

land_space/short/videos/001323.mp4: downloading bytes:           |  0.00B            

10 videos descargados de 1699 candidatos en /content/datos_roadvision/muestras/Accident-Bench__Dataset


### B) Hugging Face, videos dentro de un ZIP (sin bajar el ZIP completo)

In [ ]:
DESCARGAR = True
ZIP = zips_con_video[0] if zips_con_video else None   # tupla (repo, ruta_del_zip)
N_VIDEOS = 10

if DESCARGAR and ZIP:
    repo, ruta_zip = ZIP
    cand = sorted((m for m in contenido_zips[ZIP]
                   if m["miembro"].lower().endswith(EXT_VIDEO) and m["bytes"] <= limite_bytes
                   and pasa_filtro(m["miembro"])),
                  key=lambda m: m["miembro"])
    destino = os.path.join(DIR_MUESTRAS, repo.replace("/", "__"), os.path.basename(ruta_zip) + "_extraido")
    bajados = extraer_de_zip_hf(repo, ruta_zip, [m["miembro"] for m in muestra_espaciada(cand, N_VIDEOS)],
                                destino, detalles[repo]["revision"])
    print(f"{len(bajados)} videos extraídos de {len(cand)} candidatos en {destino}")
else:
    print("Sin extraer. Pon DESCARGAR = True para extraer la muestra.")

10 videos extraídos de 1000 candidatos en /content/datos_roadvision/muestras/kyh9191__VRU-Accident/VRU_videos.zip_extraido


### C) Kaggle con `kagglehub` (solo los archivos elegidos)

`kagglehub.dataset_download(dataset, path=archivo)` baja **un solo archivo**, no el dataset completo.

In [ ]:
DESCARGAR = True
REPO_KAGGLE = repos_kaggle[0] if repos_kaggle else None   # p. ej. "picekl/accident"
N_VIDEOS = 10

if DESCARGAR and REPO_KAGGLE:
    import kagglehub
    d = detalles[REPO_KAGGLE]
    cand = sorted((a for a in d["archivos"]
                   if (a["archivo"] or "").lower().endswith(EXT_VIDEO)
                   and (a["bytes"] is None or a["bytes"] <= limite_bytes) and pasa_filtro(a["archivo"])),
                  key=lambda a: a["archivo"])
    destino = os.path.join(DIR_MUESTRAS, REPO_KAGGLE.replace("/", "__"))
    bajados = []
    for a in muestra_espaciada(cand, N_VIDEOS):
        local = kagglehub.dataset_download(REPO_KAGGLE, path=a["archivo"])
        salida = os.path.join(destino, a["archivo"].replace("/", os.sep))
        os.makedirs(os.path.dirname(salida), exist_ok=True)
        if zipfile.is_zipfile(local) and not a["archivo"].lower().endswith(".zip"):
            with zipfile.ZipFile(local) as z:      # a veces llega comprimido
                z.extractall(os.path.dirname(salida))
        else:
            shutil.copy2(local, salida)            # del caché de kagglehub a la carpeta del proyecto
        bajados.append(salida)
    print(f"{len(bajados)} videos descargados de {len(cand)} candidatos en {destino}")
else:
    print("Sin descargar. Pon DESCARGAR = True para bajar la muestra.")

100%|██████████| 825k/825k [00:00<00:00, 2.69MB/s]


100%|██████████| 1.68M/1.68M [00:00<00:00, 3.77MB/s]


100%|██████████| 392k/392k [00:00<00:00, 1.57MB/s]


100%|██████████| 3.31M/3.31M [00:00<00:00, 7.77MB/s]


100%|██████████| 4.62M/4.62M [00:00<00:00, 7.63MB/s]


100%|██████████| 7.04M/7.04M [00:00<00:00, 11.9MB/s]


100%|██████████| 3.49M/3.49M [00:00<00:00, 7.99MB/s]


100%|██████████| 8.39M/8.39M [00:00<00:00, 17.0MB/s]


100%|██████████| 2.18M/2.18M [00:00<00:00, 5.46MB/s]


100%|██████████| 2.14M/2.14M [00:00<00:00, 4.38MB/s]

10 videos descargados de 4238 candidatos en /content/datos_roadvision/muestras/picekl__accident


## R0.6 — Resolución, FPS, duración y plan de estandarización

Se analizan **todos** los videos que haya en la carpeta de muestras, incluidas descargas de sesiones anteriores. Usa `ffprobe` (de FFmpeg) si está instalado; si no, OpenCV.

In [ ]:
def _fps_desde_fraccion(texto):
    try:
        num, den = texto.split("/")
        return round(float(num) / float(den), 3) if float(den) else None
    except Exception:
        return None


def analizar_video(ruta):
    """Resolución, FPS, frames, duración y códec."""
    if shutil.which("ffprobe"):
        cmd = ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
               "stream=width,height,r_frame_rate,nb_frames,codec_name:format=duration",
               "-of", "json", ruta]
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        if proc.returncode == 0:
            datos = json.loads(proc.stdout)
            st = (datos.get("streams") or [{}])[0]
            dur = (datos.get("format") or {}).get("duration")
            fr = st.get("nb_frames")
            return {"ruta": ruta, "ancho": st.get("width"), "alto": st.get("height"),
                    "fps": _fps_desde_fraccion(st.get("r_frame_rate", "")),
                    "frames": int(fr) if fr and fr.isdigit() else None,
                    "duracion_s": round(float(dur), 2) if dur else None,
                    "codec": st.get("codec_name"), "metodo": "ffprobe"}
    try:
        import cv2
    except ImportError:
        return {"ruta": ruta, "error": "Instala FFmpeg o opencv-python"}
    cap = cv2.VideoCapture(ruta)
    if not cap.isOpened():
        return {"ruta": ruta, "error": "No se pudo abrir el video"}
    ancho, alto = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps, frames = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
    cap.release()
    return {"ruta": ruta, "ancho": ancho, "alto": alto, "fps": round(fps, 3) if fps else None,
            "frames": frames or None, "duracion_s": round(frames / fps, 2) if fps and frames else None,
            "codec": "".join(chr((fourcc >> 8 * i) & 0xFF) for i in range(4)).strip(),
            "metodo": "opencv"}


rutas = []
for raiz, _dirs, archivos in os.walk(DIR_MUESTRAS):
    if ".cache" in raiz.split(os.sep):
        continue
    rutas += [os.path.join(raiz, f) for f in sorted(archivos) if f.lower().endswith(EXT_VIDEO)]

tabla_videos = pd.DataFrame([analizar_video(r) for r in rutas])
if tabla_videos.empty:
    print("Todavía no hay videos de muestra. Descarga algunos en R0.5.")
else:
    tabla_videos["origen"] = tabla_videos["ruta"].map(lambda r: os.path.relpath(r, DIR_MUESTRAS).split(os.sep)[0])
    # ruta dentro de su origen, para distinguir archivos con el mismo nombre en carpetas distintas
    tabla_videos["archivo"] = tabla_videos["ruta"].map(
        lambda r: os.path.join(*os.path.relpath(r, DIR_MUESTRAS).split(os.sep)[1:]))
    display(tabla_videos.drop(columns=["ruta"]))

,ancho,alto,fps,frames,duracion_s,codec,metodo,origen,archivo
0,1920,1080,20.00,331,16.55,h264,ffprobe,picekl__accident,synthetic_videos/videos/head-on/Town03_head-on_night_51.mp4
1,1920,1080,20.00,310,15.50,h264,ffprobe,picekl__accident,synthetic_videos/videos/head-on/Town06_head-on_sunset_27.mp4
2,1920,1080,20.00,311,15.55,h264,ffprobe,picekl__accident,synthetic_videos/videos/rear-end/Town05_rear-end_night_240.mp4
3,1920,1080,20.00,289,14.45,h264,ffprobe,picekl__accident,synthetic_videos/videos/rear-end/Town07_rear-end_wet_33.mp4
4,1920,1080,20.00,275,13.75,h264,ffprobe,picekl__accident,synthetic_videos/videos/sideswipe/Town06_sideswipe_wet_40.mp4
5,960,720,24.00,299,29.96,h264,ffprobe,picekl__accident,real_videos/-2UPLUV7JLg_00.mp4
6,960,720,24.00,450,30.00,h264,ffprobe,picekl__accident,real_videos/CRu-S6GbqF0_00.mp4
7,960,720,24.00,67,4.42,h264,ffprobe,picekl__accident,real_videos/QE6eIgOY5v8_00.mp4
8,1280,720,29.97,502,24.96,h264,ffprobe,picekl__accident,real_videos/cZo9DI6DSIY_02.mp4
9,1920,1080,30.00,197,24.53,h264,ffprobe,picekl__accident,real_videos/paIiPXuFj7Q_00.mp4


### Estándar propuesto para el proyecto (ajústalo en equipo)

In [ ]:
OBJ_ANCHO, OBJ_ALTO = 1280, 720
OBJ_FPS = 10
OBJ_CLIP_S = 5


def plan_estandarizacion(v, ancho, alto, fps, clip_s):
    """Qué hay que hacerle a un video para llevarlo al estándar del proyecto."""
    acciones = []
    if not v.get("ancho") or not v.get("alto") or pd.isna(v.get("ancho")):
        return {"acciones": "no se pudo leer", "clips_resultantes": 0}
    if abs(v["ancho"] / v["alto"] - ancho / alto) > 0.01:
        acciones.append("cambiar aspecto (recorte o relleno)")
    if (v["ancho"], v["alto"]) != (ancho, alto):
        acciones.append("subir resolución" if v["ancho"] < ancho else "redimensionar")
    if v.get("fps") and abs(v["fps"] - fps) > 0.01:
        acciones.append("remuestrear FPS (bajar)" if v["fps"] > fps else "remuestrear FPS (subir: ojo)")
    dur = v.get("duracion_s") or 0
    if dur < clip_s:
        acciones.append("muy corto: descartar o rellenar")
        clips = 0
    else:
        clips = int(dur // clip_s)
        if clips > 1 or dur % clip_s > 0.01:
            acciones.append(f"segmentar en {clips} clips")
    return {"acciones": "; ".join(acciones) or "ya cumple", "clips_resultantes": clips}


if tabla_videos.empty or "ancho" not in tabla_videos:
    print("Todavía no hay videos de muestra.")
else:
    planes = tabla_videos.apply(lambda r: plan_estandarizacion(r.to_dict(), OBJ_ANCHO, OBJ_ALTO, OBJ_FPS, OBJ_CLIP_S),
                                axis=1, result_type="expand")
    t = pd.concat([tabla_videos[["origen", "archivo", "ancho", "alto", "fps", "duracion_s"]], planes], axis=1)
    validos = t.dropna(subset=["ancho", "alto"])
    resumen = validos.assign(
        resolucion=validos["ancho"].astype(int).astype(str) + "x" + validos["alto"].astype(int).astype(str)
    ).groupby("origen").agg(
        videos=("archivo", "count"),
        resoluciones=("resolucion", lambda s: ", ".join(sorted(set(s)))),
        fps_min=("fps", "min"), fps_max=("fps", "max"),
        dur_min_s=("duracion_s", "min"), dur_max_s=("duracion_s", "max"),
        clips_resultantes=("clips_resultantes", "sum"),
    )
    display(Markdown("**Resumen por fuente**"))
    display(resumen)
    display(Markdown("**Qué hay que hacerle a cada video**"))
    display(t)

**Resumen por fuente**

,videos,resoluciones,fps_min,fps_max,dur_min_s,dur_max_s,clips_resultantes
origen,,,,,,,
Accident-Bench__Dataset,10,"1280x720, 480x360",10.0,30.0,5.00,250.54,70
kyh9191__VRU-Accident,10,"1280x720, 1584x660, 640x358, 640x360",20.0,20.0,4.00,20.65,16
picekl__accident,10,"1280x720, 1920x1080, 960x720",20.0,30.0,4.42,30.00,32


**Qué hay que hacerle a cada video**

,origen,archivo,ancho,alto,fps,duracion_s,acciones,clips_resultantes
0,picekl__accident,synthetic_videos/videos/head-on/Town03_head-on_night_51.mp4,1920,1080,20.00,16.55,redimensionar; remuestrear FPS (bajar); segmentar en 3 clips,3
1,picekl__accident,synthetic_videos/videos/head-on/Town06_head-on_sunset_27.mp4,1920,1080,20.00,15.50,redimensionar; remuestrear FPS (bajar); segmentar en 3 clips,3
2,picekl__accident,synthetic_videos/videos/rear-end/Town05_rear-end_night_240.mp4,1920,1080,20.00,15.55,redimensionar; remuestrear FPS (bajar); segmentar en 3 clips,3
3,picekl__accident,synthetic_videos/videos/rear-end/Town07_rear-end_wet_33.mp4,1920,1080,20.00,14.45,redimensionar; remuestrear FPS (bajar); segmentar en 2 clips,2
4,picekl__accident,synthetic_videos/videos/sideswipe/Town06_sideswipe_wet_40.mp4,1920,1080,20.00,13.75,redimensionar; remuestrear FPS (bajar); segmentar en 2 clips,2
5,picekl__accident,real_videos/-2UPLUV7JLg_00.mp4,960,720,24.00,29.96,cambiar aspecto (recorte o relleno); subir resolución; remuestrear FPS (baja...,5
6,picekl__accident,real_videos/CRu-S6GbqF0_00.mp4,960,720,24.00,30.00,cambiar aspecto (recorte o relleno); subir resolución; remuestrear FPS (baja...,6
7,picekl__accident,real_videos/QE6eIgOY5v8_00.mp4,960,720,24.00,4.42,cambiar aspecto (recorte o relleno); subir resolución; remuestrear FPS (baja...,0
8,picekl__accident,real_videos/cZo9DI6DSIY_02.mp4,1280,720,29.97,24.96,remuestrear FPS (bajar); segmentar en 4 clips,4
9,picekl__accident,real_videos/paIiPXuFj7Q_00.mp4,1920,1080,30.00,24.53,redimensionar; remuestrear FPS (bajar); segmentar en 4 clips,4


**Cómo elegir el estándar:** que la mayoría de las fuentes **baje** hacia él, no que suba. Subir resolución o FPS inventa información que no existe. Para LocateAnything la resolución importa más que los FPS, porque trabaja frame por frame; para el detector y el tracker, unos 10 fps suelen bastar.

## R0.7 — Frames clave para probar LocateAnything

LocateAnything recibe **imágenes**, no video. Esta celda saca unos cuantos frames repartidos de cada video de muestra y los guarda como JPG: son la entrada para la primera prueba del modelo, antes de cualquier LoRA.

In [ ]:
EXTRAER = True
FRAMES_POR_VIDEO = 6

if EXTRAER and not tabla_videos.empty:
    import cv2
    frames_extraidos = []
    for ruta in tabla_videos["ruta"]:
        cap = cv2.VideoCapture(ruta)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            continue
        rel = os.path.splitext(os.path.relpath(ruta, DIR_MUESTRAS))[0]
        destino = os.path.join(DIR_FRAMES, rel)
        os.makedirs(destino, exist_ok=True)
        for i in [int(total * (k + 0.5) / FRAMES_POR_VIDEO) for k in range(FRAMES_POR_VIDEO)]:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ok, img = cap.read()
            if ok:
                salida = os.path.join(destino, f"frame_{i:06d}.jpg")
                cv2.imencode(".jpg", img)[1].tofile(salida)   # evita problemas con acentos en Windows
                frames_extraidos.append(salida)
        cap.release()
    print(f"{len(frames_extraidos)} frames guardados en {DIR_FRAMES}")
else:
    print("Pon EXTRAER = True cuando ya tengas videos de muestra.")

180 frames guardados en /content/datos_roadvision/frames


## R0.8 — Modelo base, GPU y presupuesto de tiempo real

Lo que dice la ficha oficial de `nvidia/LocateAnything-3B` (verificado):

- **Entrada:** imagen + texto. **Salida:** etiquetas y cajas `<box>` con coordenadas. **No recibe video**: se usa frame por frame.
- **Arquitectura:** MoonViT + Qwen2.5-3B-Instruct + proyector MLP.
- **Licencia:** NVIDIA License, **solo investigación académica / sin fines de lucro**.
- **GPU soportadas:** NVIDIA Ampere (A100), Hopper (H100), Lovelace (L40, RTX 4090) y Blackwell, **en Linux**. MagiAttention (contexto largo) solo en Hopper/Blackwell; en Ampere/Lovelace se usa el runtime `la_flash`.
- **LoRA:** script `shell/locate-anything-lora-visual-prompt.sh` en el repo `NVlabs/Eagle` (carpeta `Embodied`): LoRA en el LLM con rango 64, backbone y LLM congelados, proyector entrenable.
- **Velocidad de referencia:** en una A100, lote de 4 imágenes 4K = 8.03 s (~2 s por imagen) con 11.7 GB de memoria reservada.

In [ ]:
def detectar_gpu():
    """Nombre, VRAM y compute capability de las GPU NVIDIA de esta máquina."""
    gpus = []
    try:
        import torch
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                p = torch.cuda.get_device_properties(i)
                gpus.append({"nombre": p.name, "vram_GB": round(p.total_memory / 1e9, 1),
                             "compute_cap": float(f"{p.major}.{p.minor}")})
            return gpus
    except Exception:
        pass
    if shutil.which("nvidia-smi"):
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                                  "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=20).stdout
            for linea in out.strip().splitlines():
                nombre, mem, cap = [x.strip() for x in linea.split(",")]
                gpus.append({"nombre": nombre, "vram_GB": round(float(mem) / 1024, 1),
                             "compute_cap": float(cap)})
        except Exception:
            pass
    return gpus


def veredicto_gpu(g):
    cc = g["compute_cap"]
    if cc >= 9.0:
        return "✅ Hopper/Blackwell: soporta todo, incluido MagiAttention."
    if cc >= 8.0:
        return "🟡 Ampere/Lovelace: inferencia con la_flash; LoRA posible sin contexto largo."
    return "❌ Anterior a Ampere (p. ej. T4, GTX): fuera de lo soportado oficialmente."


so = platform.system()
print("Sistema operativo:", so)
if so != "Linux":
    print("⚠️  La ficha solo prueba Linux. En Windows usa WSL2 con CUDA, o entrena en Colab o un servidor Linux.")

gpus = detectar_gpu()
if gpus:
    for g in gpus:
        print(f"- {g['nombre']} · {g['vram_GB']} GB · compute {g['compute_cap']} → {veredicto_gpu(g)}")
else:
    print("- No se detectó GPU NVIDIA. Opciones: Colab con L4 o A100 (la T4 gratuita queda fuera de lo "
          "soportado) o un servidor de la universidad.")

modelo = detalles.get("nvidia/LocateAnything-3B", {})
if not modelo.get("error") and modelo.get("archivos"):
    pesos = sum(a["bytes"] or 0 for a in modelo["archivos"] if a["archivo"].endswith(".safetensors"))
    print(f"\nPesos del modelo: {pesos / 1e9:.2f} GB en .safetensors (revisión {(modelo.get('revision') or '')[:10]})")

Sistema operativo: Linux
- Tesla T4 · 15.6 GB · compute 7.5 → ❌ Anterior a Ampere (p. ej. T4, GTX): fuera de lo soportado oficialmente.

Pesos del modelo: 7.66 GB en .safetensors (revisión c32291ca5e)


### Presupuesto de tiempo real: ¿por qué en cascada?

Una GPU que tarda `LATENCIA_S` segundos por imagen procesa `1 / LATENCIA_S` imágenes por segundo. Si llegan más, se hace fila. Ajusta los números y compara.

In [ ]:
N_CAMARAS = 100
FPS_CAMARA = 10
LATENCIA_S = 2.0          # mídela tú con los frames de R0.7; 2 s es la referencia con imágenes 4K
ALERTAS_POR_HORA = 2      # alertas del paso 2 por cámara por hora
FRAMES_POR_ALERTA = 3


def presupuesto_tiempo_real(camaras, fps, latencia_s, alertas_hora, frames_alerta):
    """GPUs equivalentes necesarias: todo con LocateAnything vs. en cascada."""
    directo = camaras * fps
    cascada = camaras * alertas_hora * frames_alerta / 3600
    return {"imagenes_por_s_directo": round(directo, 2), "gpus_directo": round(directo * latencia_s, 1),
            "imagenes_por_s_cascada": round(cascada, 4), "gpus_cascada": round(cascada * latencia_s, 3)}


p = presupuesto_tiempo_real(N_CAMARAS, FPS_CAMARA, LATENCIA_S, ALERTAS_POR_HORA, FRAMES_POR_ALERTA)
display(pd.DataFrame([
    {"Estrategia": "Todo con LocateAnything", "Imágenes/s": p["imagenes_por_s_directo"], "GPUs equivalentes": p["gpus_directo"]},
    {"Estrategia": "En cascada (solo alertas)", "Imágenes/s": p["imagenes_por_s_cascada"], "GPUs equivalentes": p["gpus_cascada"]},
]))
print(f"El detector del paso 1 sí tiene que aguantar {N_CAMARAS * FPS_CAMARA} imágenes/s.")

,Estrategia,Imágenes/s,GPUs equivalentes
0,Todo con LocateAnything,1000.0000,2000.000
1,En cascada (solo alertas),0.1667,0.333


El detector del paso 1 sí tiene que aguantar 1000 imágenes/s.


## R0.9 — Manifiesto de procedencia (Regla III)

Archivo, bytes y checksum de todo lo descargado. A escala de terabyte, una descarga interrumpida produce archivos truncados que parecen válidos; el checksum es la forma barata de detectarlo.

In [ ]:
def generar_manifiesto(directorio, salida, algoritmo="sha256"):
    entradas = []
    for raiz, _dirs, archivos in os.walk(directorio):
        if ".cache" in raiz.split(os.sep):
            continue
        for nombre in sorted(archivos):
            ruta = os.path.join(raiz, nombre)
            h = hashlib.new(algoritmo)
            with open(ruta, "rb") as fh:
                for bloque in iter(lambda: fh.read(1024 * 1024), b""):
                    h.update(bloque)
            entradas.append({"ruta": os.path.relpath(ruta, directorio),
                             "bytes": os.path.getsize(ruta), algoritmo: h.hexdigest()})
    manifiesto = {"directorio": os.path.abspath(directorio), "n_archivos": len(entradas),
                  "bytes_totales": sum(e["bytes"] for e in entradas), "algoritmo": algoritmo,
                  "archivos": entradas}
    with open(salida, "w", encoding="utf-8") as fh:
        json.dump(manifiesto, fh, indent=2, ensure_ascii=False)
    return manifiesto


manifiesto = generar_manifiesto(DIR_MUESTRAS, os.path.join(DIR_DATOS, "manifiesto_muestras.json"))
print({k: v for k, v in manifiesto.items() if k != "archivos"})

{'directorio': '/content/datos_roadvision/muestras', 'n_archivos': 30, 'bytes_totales': 146754186, 'algoritmo': 'sha256'}


## Ficha de candidatura

Lo marcado con **(verificar)** es una hipótesis basada en la documentación de cada dataset: confírmala con una celda de este notebook antes de copiarla a la bitácora.

| Campo | ACCIDENT (Kaggle) | Accident-Bench land_space | BMD-45 |
|---|---|---|---|
| Rol en la cascada | Paso 2 y LoRA: incidentes | Paso 2 y LoRA: incidentes extra | Paso 1: detector |
| Perspectiva | CCTV fijo (real) + sintético que lo imita | Mixta (verificar con la muestra) | CCTV fijo, 3,679 cámaras |
| Tipo de dato | Video | Video | Imágenes 1080p |
| Tamaño verificado | R0.1 | R0.1 (`GB_en_alcance`) | R0.1 |
| Formatos | R0.2 / R0.3 | R0.2 / R0.3 | R0.2 / R0.3 |
| Licencia | No indicada en su sitio (verificar en Kaggle) | Sin licencia en la tarjeta (verificar) | CC BY 4.0 |
| Resolución, FPS, duración | R0.6 | R0.6 | No aplica |
| Entidades para el join | clips ↔ anotaciones de incidente (verificar llave) | videos ↔ preguntas/respuestas por video (verificar) | imágenes ↔ cajas, por `image_id` (verificar) |
| Columna de alta cardinalidad | id de clip o de fuente (verificar) | id de video | id de cámara, ~3,679 (verificar) |
| Muestra del 0.1% | R0.5 C (Kaggle) | R0.5 A (videos sueltos) | R0.3 + descarga parcial |
| Riesgo principal | Licencia y mezcla real/sintético | Mezcla de perspectivas | No es video; ciudad muy distinta |

**El campo del join es el que más equipos dejan vacío.** Si no puedes escribir dos entidades y la llave que las une, el candidato no cumple.

## Descartados y por qué

| Dataset | Motivo |
|---|---|
| `linxxx3/bdd100k_videos` | 100% cámara de auto y ~2 TB en un ZIP partido en 19 piezas. |
| `thundarstrom/traffic-surveillance-video-corpus` | El repo solo trae el README; no hay videos. |
| `dronefreak/RDD2022`, `TamAko783/Unified_Road_Defect_Dataset` | Imágenes desde vehículo. Una cámara del C5 está demasiado alta y lejos para ver baches. Si el equipo mantiene el módulo de deterioro, va aparte y se declara que no sale del C5. |
| `UniDataPro/real-time-traffic-video-dataset`, `ud-smart-city/car-accident-video` | Solo muestra gratuita; el resto es de pago y con licencia NC-ND. |
| `Accident-Bench` → `air_space/`, `water_space/` | Aviones y barcos, fuera del dominio. |

## Lo que conviene saber desde ahora

1. **Separen train y test por cámara o por fuente, no por clip.** Si clips de la misma cámara caen en los dos lados, el modelo memoriza el fondo y las métricas salen infladas. Es el error más común en video.
2. **Los incidentes son raros.** En una cámara real, más del 99% del tiempo no pasa nada. Además de *accuracy*, reporten *precision*, *recall* y falsas alarmas por cámara por hora, que es lo que le importa a un operador del C5.
3. **Domain gap.** Ningún dataset es de Guadalajara. Decláralo como limitación y, si consiguen videos públicos del C5 (notas de noticias), úsenlos solo como prueba final, nunca para entrenar.
4. **Real vs. sintético (ACCIDENT).** Lleven la cuenta por separado y prueben siempre con clips reales.
5. **El experimento que más pesa:** LocateAnything **base contra LocateAnything con LoRA**, sobre los mismos frames de prueba. Sin esa comparación no hay forma de demostrar que el LoRA aportó algo.
6. **Licencias.** LocateAnything es solo académico (ok para la escuela, no para un producto). Verifiquen las de ACCIDENT y Accident-Bench antes de publicar resultados.
7. **Privacidad.** El CCTV muestra placas y rostros. Si publican frames en el reporte, difumínenlos.
8. **Hardware.** Revisen R0.8 antes de planear el LoRA: si no hay GPU Ampere o superior con Linux, el plan tiene que incluir Colab (L4/A100) o un servidor.